In [1]:

import pandas as pd
import duckdb

In [2]:
# Customers table
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "customer_name": ["Alex", "Maria", "James", "Sarah"],
    "state": ["CA", "CA", "TX", "NY"]
})

In [3]:
# Orders table
orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 1, 2, 3, 3],
    "amount": [200, 150, 300, 100, 250]
})

In [4]:
display(customers)
display(orders)

,customer_id,customer_name,state
0,1,Alex,CA
1,2,Maria,CA
2,3,James,TX
3,4,Sarah,NY


,order_id,customer_id,amount
0,101,1,200
1,102,1,150
2,103,2,300
3,104,3,100
4,105,3,250


In [ ]:
# example query

# query = """
# SELECT
# FROM customers AS c
# JOIN orders AS o
#     ON c.customer_id = o.customer_id

# """

# result = duckdb.sql(query).df()

# display(result)

ParserException: Parser Error: SELECT clause without selection list

In [10]:
query = """
SELECT c.state, COUNT (DISTINCT c.customer_id) AS customer_count, COUNT(o.order_id) AS order_count, SUM(o.amount) AS total_sales
FROM customers AS c
JOIN orders AS o
ON c.customer_id = o.customer_id
GROUP BY c.state
"""

result = duckdb.sql(query).df()

display(result)

,state,customer_count,order_count,total_sales
0,CA,2,3,650.0
1,TX,1,2,350.0


In [11]:
display(customers)
display(orders)

,customer_id,customer_name,state
0,1,Alex,CA
1,2,Maria,CA
2,3,James,TX
3,4,Sarah,NY


,order_id,customer_id,amount
0,101,1,200
1,102,1,150
2,103,2,300
3,104,3,100
4,105,3,250


In [ ]:
#CTE intro
query = """

WITH state_sales AS (
SELECT c.state, SUM(o.amount) AS total_sales
FROM customers AS c
JOIN orders AS o
ON c.customer_id = o.customer_id
GROUP BY c.state
)

SELECT *
FROM state_sales

"""


result = duckdb.sql(query).df()

display(result)

ParserException: Parser Error: syntax error at or near ">"

LINE 13: WHERE > 500
               ^

In [16]:
#CTE intro
query = """

WITH state_sales AS (
SELECT c.state, SUM(o.amount) AS total_sales
FROM customers AS c
JOIN orders AS o
ON c.customer_id = o.customer_id
GROUP BY c.state
)

SELECT state, total_sales
FROM state_sales
WHERE total_sales > 500

"""


result = duckdb.sql(query).df()

display(result)

,state,total_sales
0,CA,650.0


In [19]:
query = """
WITH customer_totals AS (
SELECT c.customer_id, SUM(o.amount) AS total_spent
FROM customers AS c
JOIN orders AS o
ON c.customer_id = o.customer_id
GROUP BY c.customer_id)

SELECT c.customer_name, c.state, ct.total_spent
FROM customer_totals AS ct
JOIN customers AS c
ON c.customer_id = ct.customer_id
"""

result = duckdb.sql(query).df()

display(result)


,customer_name,state,total_spent
0,Alex,CA,350.0
1,Maria,CA,300.0
2,James,TX,350.0


In [ ]:
#more than one cte in the same query
query = """
WITH first_cte AS (
SELECT c.customer_id, SUM(o.amount) AS total_spent
FROM customers AS c
JOIN orders AS o
ON c.customer_id = o.customer_id
GROUP BY c.customer_id
),
second_cte AS (
SELECT c.customer_id, COUNT(o.order_id) AS order_count
FROM customers AS c
JOIN orders AS o
ON c.customer_id = o.customer_id
GROUP BY c.customer_id
)

SELECT f.customer_id, f.total_spent, s.order_count
FROM first_cte AS f
JOIN second_cte AS s
ON f.customer_id = s.customer_id
ORDER BY f.customer_id ASC
"""

result = duckdb.sql(query).df()

display(result)

,customer_id,total_spent,order_count
0,1,350.0,2
1,2,300.0,1
2,3,350.0,2


In [23]:
display(customers)
display(orders)

,customer_id,customer_name,state
0,1,Alex,CA
1,2,Maria,CA
2,3,James,TX
3,4,Sarah,NY


,order_id,customer_id,amount
0,101,1,200
1,102,1,150
2,103,2,300
3,104,3,100
4,105,3,250


In [ ]:
#Challenge
query = """
WITH customer_total AS (
SELECT c.customer_id, SUM(o.amount) AS total_spent
FROM customers AS c
JOIN orders AS o
ON c.customer_id = o.customer_id
GROUP BY c.customer_id
)

SELECT c.customer_name, c.state, ct.total_spent
FROM customers AS c
JOIN customer_total AS ct
ON c.customer_id = ct.customer_id
WHERE ct.total_spent >= 325
ORDER BY ct.total_spent DESC
"""

result = duckdb.sql(query).df()

display(result)

,customer_name,state,total_spent
0,Alex,CA,350.0
1,James,TX,350.0


In [2]:
import pandas as pd
import numpy as np

In [3]:
sales = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105, 106, 107, 108],
    "customer": ["Alex", "Alex", "Maria", "James", "James", "Sarah", "Maria", "Alex"],
    "state": ["CA", "CA", "CA", "TX", "TX", "NY", "CA", "CA"],
    "product": ["Laptop", "Monitor", "Laptop", "Keyboard", "Monitor", "Laptop", "Keyboard", "Monitor"],
    "quantity": [1, 2, 1, 3, 1, 2, 2, 1],
    "price": [1200, 300, 1200, 100, 300, 1200, 100, 300]
})


In [29]:
display(sales)

,order_id,customer,state,product,quantity,price
0,101,Alex,CA,Laptop,1,1200
1,102,Alex,CA,Monitor,2,300
2,103,Maria,CA,Laptop,1,1200
3,104,James,TX,Keyboard,3,100
4,105,James,TX,Monitor,1,300
5,106,Sarah,NY,Laptop,2,1200
6,107,Maria,CA,Keyboard,2,100
7,108,Alex,CA,Monitor,1,300


In [31]:
#checks for inspecting a dataframe
sales.head()

,order_id,customer,state,product,quantity,price
0,101,Alex,CA,Laptop,1,1200
1,102,Alex,CA,Monitor,2,300
2,103,Maria,CA,Laptop,1,1200
3,104,James,TX,Keyboard,3,100
4,105,James,TX,Monitor,1,300


In [32]:
sales.tail()

,order_id,customer,state,product,quantity,price
3,104,James,TX,Keyboard,3,100
4,105,James,TX,Monitor,1,300
5,106,Sarah,NY,Laptop,2,1200
6,107,Maria,CA,Keyboard,2,100
7,108,Alex,CA,Monitor,1,300


In [36]:
sales.shape

(8, 6)

In [37]:
sales.columns

Index(['order_id', 'customer', 'state', 'product', 'quantity', 'price'], dtype='str')

In [38]:
sales.dtypes

order_id    int64
customer      str
state         str
product       str
quantity    int64
price       int64
dtype: object

In [39]:
sales.info

<bound method DataFrame.info of    order_id customer state   product  quantity  price
0       101     Alex    CA    Laptop         1   1200
1       102     Alex    CA   Monitor         2    300
2       103    Maria    CA    Laptop         1   1200
3       104    James    TX  Keyboard         3    100
4       105    James    TX   Monitor         1    300
5       106    Sarah    NY    Laptop         2   1200
6       107    Maria    CA  Keyboard         2    100
7       108     Alex    CA   Monitor         1    300>

In [42]:
sales['customer']

0     Alex
1     Alex
2    Maria
3    James
4    James
5    Sarah
6    Maria
7     Alex
Name: customer, dtype: str

In [ ]:
sales[['customer','price']]

,customer,price
0,Alex,1200
1,Alex,300
2,Maria,1200
3,James,100
4,James,300
5,Sarah,1200
6,Maria,100
7,Alex,300


In [45]:
sales[['customer','state','product']]

,customer,state,product
0,Alex,CA,Laptop
1,Alex,CA,Monitor
2,Maria,CA,Laptop
3,James,TX,Keyboard
4,James,TX,Monitor
5,Sarah,NY,Laptop
6,Maria,CA,Keyboard
7,Alex,CA,Monitor


In [46]:
#filter
sales[sales["state"] == "CA"]

,order_id,customer,state,product,quantity,price
0,101,Alex,CA,Laptop,1,1200
1,102,Alex,CA,Monitor,2,300
2,103,Maria,CA,Laptop,1,1200
6,107,Maria,CA,Keyboard,2,100
7,108,Alex,CA,Monitor,1,300


In [47]:
sales[sales['price'] > 500]

,order_id,customer,state,product,quantity,price
0,101,Alex,CA,Laptop,1,1200
2,103,Maria,CA,Laptop,1,1200
5,106,Sarah,NY,Laptop,2,1200


In [50]:
sales[(sales['state'] == "CA") & (sales['price'] > 500)]

,order_id,customer,state,product,quantity,price
0,101,Alex,CA,Laptop,1,1200
2,103,Maria,CA,Laptop,1,1200


In [51]:
sales[(sales['state'] == "CA") | (sales['price'] > 500)]

,order_id,customer,state,product,quantity,price
0,101,Alex,CA,Laptop,1,1200
1,102,Alex,CA,Monitor,2,300
2,103,Maria,CA,Laptop,1,1200
5,106,Sarah,NY,Laptop,2,1200
6,107,Maria,CA,Keyboard,2,100
7,108,Alex,CA,Monitor,1,300


In [52]:
#select columns AND filter rows
#df.loc[rows,columns]

sales.loc[
    sales["state"] == "CA",
    ["customer", "product", "price"]
]

,customer,product,price
0,Alex,Laptop,1200
1,Alex,Monitor,300
2,Maria,Laptop,1200
6,Maria,Keyboard,100
7,Alex,Monitor,300


In [53]:
sales.loc[
    sales["quantity"] >= 2,
    ["customer", "product", "quantity"]
]

,customer,product,quantity
1,Alex,Monitor,2
3,James,Keyboard,3
5,Sarah,Laptop,2
6,Maria,Keyboard,2


In [54]:
#sort
sales.sort_values('quantity',ascending=False)

,order_id,customer,state,product,quantity,price
3,104,James,TX,Keyboard,3,100
1,102,Alex,CA,Monitor,2,300
6,107,Maria,CA,Keyboard,2,100
5,106,Sarah,NY,Laptop,2,1200
2,103,Maria,CA,Laptop,1,1200
0,101,Alex,CA,Laptop,1,1200
4,105,James,TX,Monitor,1,300
7,108,Alex,CA,Monitor,1,300


In [5]:
sales.loc[(sales["state"] == "CA") & (sales["quantity"] >= 2), ["customer", "product", "quantity", "price"]].sort_values("price", ascending=False)

,customer,product,quantity,price
1,Alex,Monitor,2,300
6,Maria,Keyboard,2,100


In [8]:
sales.loc[((sales["state"] == "CA") | (sales["state"] == "TX")) & (sales["quantity"] >= 2), ["customer", "state", "product", "quantity", "price"]].sort_values(["quantity", "price"], ascending=[False,False])

,customer,state,product,quantity,price
3,James,TX,Keyboard,3,100
1,Alex,CA,Monitor,2,300
6,Maria,CA,Keyboard,2,100


In [9]:
sales.loc[
    (sales["state"].isin(["CA", "TX"])) &
    (sales["quantity"] >= 2),
    ["customer", "state", "product", "quantity", "price"]
].sort_values(
    ["quantity", "price"],
    ascending=[False, False]
)

,customer,state,product,quantity,price
3,James,TX,Keyboard,3,100
1,Alex,CA,Monitor,2,300
6,Maria,CA,Keyboard,2,100
